# Ejercicio 2.2 · primer índice persistente sobre tu propio corpus

Migramos el buscador del ejercicio 1.1 (NumPy, todo en memoria) a **ChromaDB**: una base vectorial que guarda los vectores **en disco**, permite añadir **metadatos** y **filtrar** las búsquedas.

Corpus: 5.000 noticias de AG News (en inglés, 4 categorías).

Plan del notebook:

| Paso | Qué hacemos |
|---|---|
| 1-2 | Dependencias y corpus |
| 3 | Decidir el schema de metadatos |
| 4 | Calcular los embeddings |
| 5 | Crear la colección persistente e insertar en batches de 500 |
| 6-7 | Funciones de búsqueda con filtros y pruebas |
| 8 | Comparar con el k-NN de NumPy |
| 9 | Mantenimiento: añadir, actualizar, borrar |
| 10 | Persistencia real |
| Bonus 1-3 | Embeddings de OpenAI, benchmark de inserción, uso de espacio |

## 1. Dependencias

Se instalan con `uv` (nunca con `pip`). Ya están declaradas en `pyproject.toml`, así que normalmente no hace falta ejecutar nada. Si falta alguna:

```bash
uv add chromadb sentence-transformers datasets
```

- `chromadb`: la base vectorial. Guarda vectores, textos y metadatos, y busca los más parecidos.
- `sentence-transformers`: el modelo que convierte texto en vectores (el mismo del ejercicio 1.1).
- `datasets`: descarga el corpus de AG News desde Hugging Face.
- `numpy`: lo usamos en el paso 8 para comparar con el buscador del ejercicio 1.1.

In [1]:
import os                 # para recorrer carpetas y medir su tamaño (Bonus 3)
import shutil             # para borrar carpetas de prueba
import subprocess         # para lanzar un Python nuevo (pasos 8 y 10)
import sys                # sys.executable = el Python de nuestro .venv
import time               # para medir cuánto tarda cada cosa
from datetime import date, timedelta   # para inventar fechas (paso 3)

import numpy as np
import chromadb
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

W1116 12:34:49.098000 42496 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Corpus: 5.000 noticias de AG News

`load_dataset` descarga el dataset la primera vez y lo deja en caché. `split="train[:5000]"` significa «de la parte de entrenamiento, quédate solo con las 5.000 primeras filas».

> En el enunciado aparece `"ag_news"`, pero con esta versión de `datasets` hay que usar el nombre completo del repositorio: `"fancyzhx/ag_news"`.

Cada noticia tiene dos campos:

- `text`: el texto de la noticia.
- `label`: la categoría, como número (0, 1, 2, 3). Los nombres de las categorías están en `dataset.features["label"].names`.

In [2]:
dataset = load_dataset("fancyzhx/ag_news", split="train[:5000]")
print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


Pasamos el dataset a dos listas paralelas con un bucle `for` (como en el ejercicio 1.1): `textos[i]` y `categorias[i]` son de la misma noticia. Convertimos el número de la etiqueta en su nombre, que es más legible (`2` → `"Business"`).

In [3]:
nombres_etiquetas = dataset.features["label"].names
print(nombres_etiquetas)

textos = []       # el texto de cada noticia
categorias = []   # la categoría de cada noticia, en el mismo orden

for noticia in dataset:
    textos.append(noticia["text"])
    numero_etiqueta = noticia["label"]
    categorias.append(nombres_etiquetas[numero_etiqueta])

print(len(textos), "noticias")

# Cuántas noticias hay de cada categoría
for nombre in nombres_etiquetas:
    print(f"   {nombre:10} {categorias.count(nombre)}")

['World', 'Sports', 'Business', 'Sci/Tech']
5000 noticias
   World      1235
   Sports     1032
   Business   1236
   Sci/Tech   1497


## 3. Schema de metadatos

Un **metadato** es un dato que acompaña a cada documento y sirve para **filtrar** la búsqueda («solo deportes», «solo desde junio de 2024»). Decidimos qué guardar para cada noticia:

| Metadato | Ejemplo | De dónde sale |
|---|---|---|
| `categoria` | `"Sports"` | la etiqueta del dataset |
| `fecha` | `20240615` | inventada (el dataset no trae fechas) |
| `longitud` | `120` | `len(texto)`, caracteres |
| `idioma` | `"en"` | lo sabemos: AG News está en inglés |
| `fuente` | `"ag_news"` | el nombre del dataset |

### ⚠️ La fecha, como número

ChromaDB solo permite comparar con `>`, `<`, `>=`... los metadatos que son **números**. Un texto como `"2024-06-15"` serviría para «es igual a», pero no para «es posterior a». Por eso guardamos la fecha como un entero con formato `AAAAMMDD`:

- `2024-06-15` → `20240615`
- `2025-01-01` → `20250101` (es mayor, y eso nos vale para ordenar por fecha).

### Los ids

Cada documento necesita un **id único** (texto). Usamos `doc_0000`, `doc_0001`... Los ids luego sirven para actualizar o borrar un documento concreto.

In [4]:
ids = []         # un id único por noticia
metadatos = []   # un diccionario de metadatos por noticia

fecha_inicial = date(2024, 1, 1)

for i in range(len(textos)):
    # f"{i:04d}" rellena con ceros hasta 4 cifras: 7 -> "0007"
    id_documento = f"doc_{i:04d}"

    # Fecha inventada: repartimos las noticias a lo largo de 2 años (730 días)
    fecha = fecha_inicial + timedelta(days=i % 730)
    fecha_como_numero = int(fecha.strftime("%Y%m%d"))   # 2024-06-15 -> 20240615

    metadato = {
        "categoria": categorias[i],
        "fecha": fecha_como_numero,
        "longitud": len(textos[i]),
        "idioma": "en",
        "fuente": "ag_news",
    }

    ids.append(id_documento)
    metadatos.append(metadato)

print(ids[0], metadatos[0])
print(ids[-1], metadatos[-1])

doc_0000 {'categoria': 'Business', 'fecha': 20240101, 'longitud': 144, 'idioma': 'en', 'fuente': 'ag_news'}
doc_4999 {'categoria': 'Business', 'fecha': 20250911, 'longitud': 289, 'idioma': 'en', 'fuente': 'ag_news'}


## 4. Embeddings de las 5.000 noticias

Usamos el mismo modelo que en el ejercicio 1.1. Es multilingüe (entiende inglés y español), así que podremos buscar en español sobre noticias en inglés.

**Calculamos nosotros los embeddings** y se los pasamos a ChromaDB ya hechos. Si no, Chroma usaría su propio modelo por defecto y no sabríamos exactamente con cuál trabaja. Además así podemos medir por separado «calcular vectores» (lento, lo hace el modelo) y «guardarlos» (lo hace Chroma).

`batch_size=64` hace que el modelo procese 64 textos a la vez, que es más rápido que uno a uno. En CPU, 5.000 noticias tardan un rato: esta es la celda más lenta del notebook.

In [5]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
# time.time() da la hora actual en segundos; restando dos horas sabemos cuánto tardó
inicio = time.time()
embeddings = model.encode(textos, batch_size=64, show_progress_bar=True)
segundos_embeddings = time.time() - inicio

print(embeddings.shape)   # (5000, 384): una fila por noticia
print(f"Calcular los embeddings: {segundos_embeddings:.1f} s")

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

(5000, 384)
Calcular los embeddings: 6.9 s


## 5. Colección persistente e inserción en batches

### Cliente persistente

`chromadb.PersistentClient(path="./chroma_db")` abre una base de datos **guardada en la carpeta `./chroma_db`**. Si la carpeta no existe, la crea. Todo lo que insertemos se escribe ahí y sigue existiendo cuando cerremos Python.

(Hay también `chromadb.Client()`, que guarda todo en memoria y lo pierde al cerrar. Es el equivalente a nuestra matriz de NumPy.)

### Colección y métrica

Una **colección** es como una «tabla» de la base: contiene documentos, vectores y metadatos. Al crearla hay que decidir la **métrica de distancia**, y luego no se puede cambiar (justo la trampa de la que hablábamos antes). Usamos **coseno** con `metadata={"hnsw:space": "cosine"}`.

Ojo: Chroma devuelve **distancias**, no similitudes. Con coseno, `distancia = 1 - similitud`. Distancia 0 = idénticos. Por eso en las funciones de búsqueda calcularemos `similitud = 1 - distancia`, para ver los mismos números que en el ejercicio 1.1.

### Insertar en batches de 500

En vez de llamar a `add` 5.000 veces (una por noticia), la llamamos 10 veces con 500 noticias cada vez. Cada llamada tiene un coste fijo (abrir transacción, escribir en disco...), y con batches ese coste se paga 10 veces en lugar de 5.000. En el Bonus 2 medimos la diferencia exacta.

`range(0, 5000, 500)` da 0, 500, 1000... 4500: el punto de inicio de cada batch. Con `[desde:hasta]` cortamos cada lista en el trozo que toca.

In [7]:
cliente = chromadb.PersistentClient(path="./chroma_db")

# Si ejecutas esta celda dos veces, borramos la colección anterior para no duplicar datos.
# try/except: si la colección no existe todavía, delete_collection da error y lo ignoramos.
try:
    cliente.delete_collection("noticias")
except Exception:
    pass

coleccion = cliente.create_collection(
    name="noticias",
    metadata={"hnsw:space": "cosine"},   # métrica coseno (fija para esta colección)
)
print("Colección creada:", coleccion.name)

Colección creada: noticias


In [8]:
TAMANO_BATCH = 500

inicio = time.time()

for desde in range(0, len(textos), TAMANO_BATCH):
    hasta = desde + TAMANO_BATCH

    coleccion.add(
        ids=ids[desde:hasta],
        documents=textos[desde:hasta],      # el texto original (Chroma lo guarda también)
        embeddings=embeddings[desde:hasta], # los vectores que calculamos nosotros
        metadatas=metadatos[desde:hasta],
    )
    print(f"   insertadas {min(hasta, len(textos))} / {len(textos)}")

segundos_insercion = time.time() - inicio
print(f"Documentos en la colección: {coleccion.count()}")
print(f"Insertar en batches de {TAMANO_BATCH}: {segundos_insercion:.2f} s")

   insertadas 500 / 5000
   insertadas 1000 / 5000
   insertadas 1500 / 5000
   insertadas 2000 / 5000
   insertadas 2500 / 5000
   insertadas 3000 / 5000
   insertadas 3500 / 5000
   insertadas 4000 / 5000
   insertadas 4500 / 5000
   insertadas 5000 / 5000
Documentos en la colección: 5000
Insertar en batches de 500: 3.03 s


## 6. Funciones de búsqueda con filtros opcionales

Una sola función hace el trabajo y las otras dos solo preparan el filtro:

| Función | Qué hace |
|---|---|
| `buscar(query, k=5, filtro=None)` | búsqueda semántica; si hay `filtro`, solo entre los documentos que lo cumplen |
| `buscar_por_categoria(query, categoria, k=5)` | prepara el filtro `categoria == X` y llama a `buscar` |
| `buscar_recientes(query, desde_fecha, k=5)` | prepara el filtro `fecha >= X` y llama a `buscar` |

### Cómo se escribe un filtro (`where`)

Es un diccionario:

```python
{"categoria": "Sports"}                    # categoria == "Sports"
{"fecha": {"$gte": 20240601}}              # fecha >= 20240601
{"fecha": {"$lt": 20250101}}               # fecha <  20250101
{"categoria": {"$in": ["Sports", "World"]}}   # categoria es una de esas
{"$and": [ {...}, {...} ]}                 # las dos condiciones a la vez
```

Operadores de comparación: `$eq` (igual), `$ne` (distinto), `$gt` (mayor), `$gte` (mayor o igual), `$lt` (menor), `$lte` (menor o igual), `$in`, `$nin`.

### Qué devuelve `collection.query`

Devuelve un diccionario con listas **dentro de otra lista**: una lista por cada query que enviamos. Como enviamos una sola, siempre usamos `[0]` para quedarnos con la lista de esa única query:

```python
resultados["documents"][0]   # los k textos
resultados["distances"][0]   # las k distancias
resultados["metadatas"][0]   # los k diccionarios de metadatos
resultados["ids"][0]         # los k ids
```

Importante: el filtro se aplica **durante** la búsqueda, no después. Si pides `k=5` con un filtro, Chroma te devuelve 5 documentos que lo cumplen (si hay suficientes), no «los 5 mejores y luego quita los que no valen».

In [9]:
def buscar(query, k=5, filtro=None):
    # 1. Convertimos la query en vector con el MISMO modelo que usamos para el corpus
    query_embedding = model.encode([query])

    # 2. Chroma busca los k vectores más cercanos (solo entre los que cumplen el filtro)
    resultados = coleccion.query(
        query_embeddings=query_embedding,
        n_results=k,
        where=filtro,   # None = sin filtro
    )

    # 3. [0] porque enviamos una sola query (ver explicación de arriba)
    textos_encontrados = resultados["documents"][0]
    distancias = resultados["distances"][0]
    metadatos_encontrados = resultados["metadatas"][0]

    print("🔎", query, "| filtro:", filtro)
    for i in range(len(textos_encontrados)):
        similitud = 1 - distancias[i]   # con coseno: distancia = 1 - similitud
        categoria = metadatos_encontrados[i]["categoria"]
        fecha = metadatos_encontrados[i]["fecha"]
        texto_corto = textos_encontrados[i][:85]
        print(f"   {similitud:.3f}  [{categoria:8}] {fecha}  {texto_corto}")
    print()

    return resultados


def buscar_por_categoria(query, categoria, k=5):
    filtro = {"categoria": categoria}
    return buscar(query, k, filtro)


def buscar_recientes(query, desde_fecha, k=5):
    # Aceptamos "2025-01-01" (cómodo para escribir) y lo pasamos a 20250101 (como guardamos la fecha)
    fecha_como_numero = int(desde_fecha.replace("-", ""))
    filtro = {"fecha": {"$gte": fecha_como_numero}}   # $gte = mayor o igual
    return buscar(query, k, filtro)

## 7. Pruebas con queries reales

Primero sin filtro, para ver el comportamiento normal. Fíjate en que la última query está en **español** y el corpus en inglés: el modelo es multilingüe.

In [10]:
buscar("oil prices hit a record high")
buscar("who won the football championship final?")
buscar("a new computer virus is spreading on the internet")
buscar("el presidente anuncia nuevas elecciones")

🔎 oil prices hit a record high | filtro: None
   0.848  [Business] 20240202  Oil Prices Hit Record (Reuters) Reuters - Oil prices jumped to a new record\high near
   0.843  [Business] 20251120  Oil Above  #36;47 After New Record (Reuters) Reuters - Oil prices struck another reco
   0.836  [Business] 20240829  Oil Hits New Record as Demand Soars  LONDON (Reuters) - Oil prices struck a fresh rec
   0.788  [Business] 20241112  Oil prices climb again tonight Oil prices moved back towards record highs tonight ami
   0.788  [Business] 20251203  Oil Above \$47 After New Record SINGAPORE (Reuters) - Oil prices struck another recor

🔎 who won the football championship final? | filtro: None
   0.498  [Sports  ] 20251226  Russia #39;s Korzhanenko wins women #39;s shot put; American favorites eliminated US 
   0.447  [Sports  ] 20251004  Germany Paddles to Bronze The men #39;s single slalom canoe race ended with balled fi
   0.444  [Sports  ] 20240703  Old college try from Moreland Earthwind Morel

{'ids': [['doc_4653', 'doc_3261', 'doc_1719', 'doc_0554', 'doc_0090']],
 'embeddings': None,
 'documents': [['Peres calls for early general election The leader of the opposition Labour party in Israel, Shimon Peres, has called for an early general election. ',
   'Results of Presidential Campaign Polls (AP) AP - Results of recent polls on the presidential race. Listed above each set of results is the name of the 2000 winner in a given state, the organization that conducted the poll, the dates, the number interviewed, whether it was adults, registered voters (RV) or likely voters (LV) and the margin of error (MoE). Results may not total 100 percent because of rounding.',
   'Interactive Web Sites Provide New Approach to Election Coverage (Editor and Publisher) Editor and Publisher - NEW YORK While most newspapers are utilizing their Web sites to supplement coverage of the presidential race, the Los Angeles Times and The New York Times have taken the frenzy over electoral votes and undec

### Filtro por categoría

Comprobamos que **todos** los resultados son de la categoría pedida. Un bucle recorre los metadatos devueltos y cuenta los que no cumplen.

Ojo a la segunda prueba: preguntamos por bolsa pero obligamos a `Sports`. El filtro manda: Chroma devuelve lo menos malo **dentro** de deportes, con similitudes bajas. El filtro no sabe si el resultado es bueno, solo lo restringe.

In [11]:
resultados = buscar_por_categoria("new tablet and smartphone launch", "Sci/Tech")

fallos = 0
for metadato in resultados["metadatas"][0]:
    if metadato["categoria"] != "Sci/Tech":
        fallos = fallos + 1
print("Resultados de otra categoría:", fallos)   # debe ser 0
print()

# Query de bolsa, pero forzando la categoría Sports
resultados = buscar_por_categoria("stock market falls as investors worry", "Sports")

fallos = 0
for metadato in resultados["metadatas"][0]:
    if metadato["categoria"] != "Sports":
        fallos = fallos + 1
print("Resultados de otra categoría:", fallos)   # debe ser 0

🔎 new tablet and smartphone launch | filtro: {'categoria': 'Sci/Tech'}
   0.518  [Sci/Tech] 20241026  Motorola and HP in Linux tie-up Motorola plans first use of the open-source software 
   0.472  [Sci/Tech] 20250711  A Nokia Smartphone With An Edge NEW YORK - It #39;s a little early to tell which came
   0.442  [Sci/Tech] 20240427  Lebanese shun mobile phones for a day in protest at high costs (AFP) AFP - Lebanese m
   0.441  [Sci/Tech] 20251002  Samsung streams on Sprint with MM-A700 Samsung #39;s latest phone offers streaming vi
   0.417  [Sci/Tech] 20251009  PalmOne Unveils SD Wi-Fi Card (PC World) PC World - Tiny 802.11b adapter will work wi

Resultados de otra categoría: 0

🔎 stock market falls as investors worry | filtro: {'categoria': 'Sports'}
   0.311  [Sports  ] 20251015  Men #39;s volleyball: US falls to Russia Athens, Greece (Sports Network) - The United
   0.292  [Sports  ] 20250816  U.S. Basketball Team Loses to Puerto Rico  ATHENS (Reuters) - The United States lost 
  

### Filtro por fecha

Misma comprobación: ningún resultado puede tener una fecha anterior a la pedida.

In [12]:
resultados = buscar_recientes("oil prices and the economy", "2025-06-01")

fallos = 0
for metadato in resultados["metadatas"][0]:
    if metadato["fecha"] < 20250601:
        fallos = fallos + 1
print("Resultados anteriores a 2025-06-01:", fallos)   # debe ser 0

🔎 oil prices and the economy | filtro: {'fecha': {'$gte': 20250601}}
   0.659  [Business] 20250614  Oil Prices Are at New High, Over \$49  LONDON (Reuters) - Oil prices raced to fresh h
   0.652  [Business] 20251120  Oil Above  #36;47 After New Record (Reuters) Reuters - Oil prices struck another reco
   0.651  [Business] 20250810  Economies Not Yet Dented by Oil Prices (Reuters) Reuters - OPEC oil producers and Cha
   0.630  [Business] 20250901  Oil Prices Hit New High The price of this key commodity has risen by about a third ju
   0.628  [Business] 20250704  Oil Tops \$49 On Renewed Iraq Violence LONDON (Reuters) - Global oil prices raced to 

Resultados anteriores a 2025-06-01: 0


### Dos filtros a la vez

Con `$and` combinamos condiciones: noticias de `Business` **y** de 2025 en adelante. Aquí usamos `buscar` directamente, pasándole el filtro.

In [13]:
filtro_combinado = {
    "$and": [
        {"categoria": "Business"},
        {"fecha": {"$gte": 20250101}},
    ]
}
resultados = buscar("oil prices and the economy", k=5, filtro=filtro_combinado)

🔎 oil prices and the economy | filtro: {'$and': [{'categoria': 'Business'}, {'fecha': {'$gte': 20250101}}]}
   0.676  [Business] 20250427  Oil at New Record  LONDON (Reuters) - Oil prices surged to a new high above  \$47 a b
   0.659  [Business] 20250614  Oil Prices Are at New High, Over \$49  LONDON (Reuters) - Oil prices raced to fresh h
   0.653  [Business] 20250329  Oil at New High Over \$49  LONDON (Reuters) - Oil prices raced to fresh highs on  Fri
   0.652  [Business] 20251120  Oil Above  #36;47 After New Record (Reuters) Reuters - Oil prices struck another reco
   0.651  [Business] 20250810  Economies Not Yet Dented by Oil Prices (Reuters) Reuters - OPEC oil producers and Cha



## 8. Comparación con el k-NN de NumPy (ejercicio 1.1)

Construimos el buscador del ejercicio 1.1 sobre **los mismos 5.000 vectores** (`embeddings`): normalizar, producto escalar con `@`, `np.argsort`. Y comparamos tres cosas:

1. **¿Mismos resultados?**
2. **¿Cuánto tarda cada una por query?**
3. **¿Cuánto cuesta «arrancar» cada una en una sesión nueva?**

Una pista antes de mirar los números: Chroma usa un índice **aproximado** (HNSW, un tipo de ANN) que busca *casi* el mejor resultado sin comparar con todos. NumPy compara con **todos** (búsqueda exacta). Con 5.000 vectores comparar con todos es trivial; el índice aproximado empieza a ganar con cientos de miles o millones.

In [14]:
def normalizar(matriz):
    # Igual que en el ejercicio 1.1: dejar cada vector con longitud 1
    longitudes = np.linalg.norm(matriz, axis=-1, keepdims=True)
    return matriz / longitudes

embeddings_norm = normalizar(embeddings)   # una sola vez


def buscar_numpy(query_norm, k=5):
    similitudes = embeddings_norm @ query_norm            # 5000 similitudes de golpe
    posiciones = np.argsort(similitudes)[::-1][:k]        # las k más parecidas
    return posiciones

### 8.1 ¿Mismos resultados?

Para cada query sacamos los 5 mejores ids con NumPy y con Chroma y comparamos las listas. Como Chroma es aproximado, podría fallar alguno, pero con 5.000 documentos lo normal es que coincidan.

In [15]:
queries_prueba = [
    "oil prices hit a record high",
    "who won the football championship final?",
    "a new computer virus is spreading on the internet",
    "el presidente anuncia nuevas elecciones",
]

for query in queries_prueba:
    query_embedding = model.encode([query])
    query_norm = normalizar(query_embedding)[0]

    # NumPy: posiciones -> ids
    posiciones = buscar_numpy(query_norm, k=5)
    ids_numpy = []
    for posicion in posiciones:
        ids_numpy.append(ids[posicion])

    # Chroma
    resultados = coleccion.query(query_embeddings=query_embedding, n_results=5)
    ids_chroma = resultados["ids"][0]

    print(query)
    print("   NumPy :", ids_numpy)
    print("   Chroma:", ids_chroma)
    print("   ¿Mismos ids y mismo orden?", ids_numpy == ids_chroma)

oil prices hit a record high
   NumPy : ['doc_0762', 'doc_3609', 'doc_3891', 'doc_1776', 'doc_3622']
   Chroma: ['doc_0762', 'doc_3609', 'doc_3891', 'doc_1776', 'doc_3622']
   ¿Mismos ids y mismo orden? True
who won the football championship final?
   NumPy : ['doc_2915', 'doc_2832', 'doc_2374', 'doc_2804', 'doc_3651']
   Chroma: ['doc_2915', 'doc_2832', 'doc_2374', 'doc_2804', 'doc_3651']
   ¿Mismos ids y mismo orden? True
a new computer virus is spreading on the internet
   NumPy : ['doc_1752', 'doc_4359', 'doc_2172', 'doc_0332', 'doc_2072']
   Chroma: ['doc_1752', 'doc_4359', 'doc_2172', 'doc_0332', 'doc_2072']
   ¿Mismos ids y mismo orden? True
el presidente anuncia nuevas elecciones
   NumPy : ['doc_4653', 'doc_3261', 'doc_1719', 'doc_0554', 'doc_0090']
   Chroma: ['doc_4653', 'doc_3261', 'doc_1719', 'doc_0554', 'doc_0090']
   ¿Mismos ids y mismo orden? True


### 8.2 Velocidad por query

Usamos los vectores de las 200 primeras noticias como si fueran queries, para medir **solo la búsqueda** (sin el tiempo de convertir texto en vector, que es igual en los dos casos y es lo más lento).

In [16]:
NUMERO_QUERIES = 200

# --- NumPy
inicio = time.time()
for i in range(NUMERO_QUERIES):
    buscar_numpy(embeddings_norm[i], k=5)
ms_numpy = (time.time() - inicio) / NUMERO_QUERIES * 1000   # milisegundos por query

# --- Chroma
inicio = time.time()
for i in range(NUMERO_QUERIES):
    # [vector]: query espera una lista de queries, aunque sea una sola
    coleccion.query(query_embeddings=[embeddings[i]], n_results=5)
ms_chroma = (time.time() - inicio) / NUMERO_QUERIES * 1000

print(f"NumPy : {ms_numpy:.2f} ms por query")
print(f"Chroma: {ms_chroma:.2f} ms por query")

NumPy : 0.30 ms por query
Chroma: 1.20 ms por query


### 8.3 Arrancar en una sesión nueva

Con NumPy, si cierras el notebook, la matriz `embeddings` se pierde y hay que **volver a calcular los 5.000 embeddings** (`segundos_embeddings`, la celda lenta del paso 4). Con Chroma, los vectores ya están en disco: solo hay que abrir la colección.

Para medirlo de verdad lanzamos un **Python nuevo** (un proceso aparte, como si cerrases todo y empezaras de cero) que abre `./chroma_db` y cuenta los documentos. `subprocess.run` ejecuta ese Python y `capture_output=True` recoge lo que imprime.

In [17]:
def ejecutar_en_python_nuevo(codigo):
    # sys.executable = el Python de nuestro .venv; "-c" = ejecutar este texto como código
    proceso = subprocess.run([sys.executable, "-c", codigo], capture_output=True, text=True)
    if proceso.returncode != 0:
        return "ERROR: " + proceso.stderr
    return proceso.stdout


codigo_abrir_coleccion = """
import time
inicio = time.time()
import chromadb
cliente = chromadb.PersistentClient(path="./chroma_db")
coleccion = cliente.get_collection("noticias")
total = coleccion.count()
print(total, "documentos listos en", round(time.time() - inicio, 2), "s (incluye importar chromadb)")
"""

print("Chroma en Python nuevo:", ejecutar_en_python_nuevo(codigo_abrir_coleccion))
print(f"NumPy en Python nuevo: hay que recalcular los embeddings → {segundos_embeddings:.1f} s")

Chroma en Python nuevo: 5000 documentos listos en 1.26 s (incluye importar chromadb)

NumPy en Python nuevo: hay que recalcular los embeddings → 6.9 s


## 9. Operaciones de mantenimiento

Tres funciones, una por operación:

| Función | Método de Chroma | Qué hace |
|---|---|---|
| `anadir_documento(...)` | `collection.add` | inserta un documento nuevo (con su embedding y metadatos) |
| `actualizar_categoria(id, nueva)` | `collection.update` | cambia un metadato de un documento, buscándolo por su id |
| `borrar_categoria(categoria)` | `collection.delete(where=...)` | borra **todos** los documentos que cumplan el filtro |

`collection.get(ids=[...])` sirve para leer un documento concreto por su id (lo usamos para comprobar).

Para no estropear el corpus real, probamos con documentos de **categorías inventadas** (`Prueba_A`, `Prueba_B`) y al final lo dejamos todo como estaba: 5.000 documentos.

In [18]:
def anadir_documento(id_documento, texto, categoria, fecha_como_numero):
    # El embedding se calcula con el mismo modelo que el resto del corpus
    embedding_nuevo = model.encode([texto])

    metadato_nuevo = {
        "categoria": categoria,
        "fecha": fecha_como_numero,
        "longitud": len(texto),
        "idioma": "en",
        "fuente": "manual",
    }

    coleccion.add(
        ids=[id_documento],
        documents=[texto],
        embeddings=embedding_nuevo,
        metadatas=[metadato_nuevo],
    )


def actualizar_categoria(id_documento, nueva_categoria):
    # Leemos los metadatos actuales, cambiamos solo la categoría y los guardamos
    documento = coleccion.get(ids=[id_documento])
    metadato = documento["metadatas"][0]
    metadato["categoria"] = nueva_categoria

    coleccion.update(ids=[id_documento], metadatas=[metadato])


def borrar_categoria(categoria):
    # where=... es el mismo tipo de filtro que usamos al buscar
    coleccion.delete(where={"categoria": categoria})

In [19]:
print("Documentos al empezar:", coleccion.count())

# 1. Añadir dos documentos nuevos en la categoría inventada "Prueba_A"
anadir_documento("doc_nuevo_1", "A new tool lets teachers build quizzes in seconds.", "Prueba_A", 20250301)
anadir_documento("doc_nuevo_2", "The city opens a new bike lane along the river.", "Prueba_A", 20250302)
print("Tras añadir 2:", coleccion.count())

# 2. Actualizar la categoría del primero: Prueba_A -> Prueba_B
actualizar_categoria("doc_nuevo_1", "Prueba_B")
documento = coleccion.get(ids=["doc_nuevo_1"])
print("Categoría de doc_nuevo_1 ahora:", documento["metadatas"][0]["categoria"])

# 3. Borrar todos los de Prueba_A (solo queda doc_nuevo_2 con esa categoría)
borrar_categoria("Prueba_A")
print("Tras borrar Prueba_A:", coleccion.count())

# 4. Limpieza: borramos también Prueba_B para dejar el corpus original
borrar_categoria("Prueba_B")
print("Tras borrar Prueba_B:", coleccion.count())   # debe volver a 5000

Documentos al empezar: 5000
Tras añadir 2: 5002
Categoría de doc_nuevo_1 ahora: Prueba_B
Tras borrar Prueba_A: 5001
Tras borrar Prueba_B: 5000


## 10. Persistencia en la práctica

Probamos que los datos sobreviven. Lanzamos otra vez un **Python nuevo**, que no tiene nada en memoria: ni el modelo, ni `embeddings`, ni `coleccion`. Solo abre la carpeta `./chroma_db`, cuenta los documentos y lee uno con su texto y metadatos.

`collection.get(where=..., limit=1)` es como `query` pero sin buscar por significado: solo filtra y devuelve documentos.

In [20]:
codigo_leer_coleccion = """
import chromadb
cliente = chromadb.PersistentClient(path="./chroma_db")
coleccion = cliente.get_collection("noticias")
print("Documentos:", coleccion.count())

un_documento = coleccion.get(where={"categoria": "Sports"}, limit=1)
print("Un documento de Sports:")
print("   id:       ", un_documento["ids"][0])
print("   metadatos:", un_documento["metadatas"][0])
print("   texto:    ", un_documento["documents"][0][:90])
"""

print(ejecutar_en_python_nuevo(codigo_leer_coleccion))

Documentos: 5000
Un documento de Sports:
   id:        doc_0448
   metadatos: {'fecha': 20250324, 'longitud': 389, 'idioma': 'en', 'fuente': 'ag_news', 'categoria': 'Sports'}
   texto:     Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying f



### Prueba manual (hazla tú)

Para sentirlo de verdad:

1. Cierra el notebook y el kernel (o cierra VS Code).
2. Abre un **notebook nuevo** en la misma carpeta `notebooks/`.
3. Ejecuta solo esto:

```python
import chromadb

cliente = chromadb.PersistentClient(path="./chroma_db")
coleccion = cliente.get_collection("noticias")
print(coleccion.count())    # 5000
```

Si ves `5000`, no hace falta volver a calcular nada. (La ruta `./chroma_db` es relativa: el notebook nuevo debe estar en la misma carpeta, `notebooks/`, para encontrar la base.)

Si quieres buscar con texto en ese notebook nuevo, solo necesitas cargar el modelo (`SentenceTransformer(...)`) para convertir la query en vector: los 5.000 vectores del corpus ya están guardados.

## Bonus 1 · embeddings de OpenAI

Creamos una segunda colección, `noticias_openai`, con embeddings de `text-embedding-3-small` en vez de `sentence-transformers`. Comparamos resultados y coste.

Diferencias importantes:

| | sentence-transformers (MiniLM) | OpenAI `text-embedding-3-small` |
|---|---|---|
| Dónde corre | en tu ordenador | en la API de OpenAI (necesita internet y clave) |
| Dimensiones | 384 | 1536 |
| Coste | gratis (pero tarda en CPU) | de pago, por token |

**Los vectores de las dos colecciones no se pueden mezclar:** tienen distinto número de dimensiones y viven en «mapas» distintos. Por eso es otra colección. Y las queries a `noticias_openai` también hay que convertirlas con OpenAI, no con MiniLM.

`load_dotenv()` lee el archivo `.env` y carga `OPENAI_API_KEY` para que `OpenAI()` la encuentre sin que la clave aparezca escrita en el código. La API acepta lotes de textos, así que mandamos 500 por llamada.

La API devuelve en `respuesta.usage.total_tokens` cuántos tokens ha gastado; con eso calculamos el coste. Precio de `text-embedding-3-small`: **0,02 USD por millón de tokens** (comprueba la web de OpenAI por si ha cambiado).

In [21]:
from dotenv import load_dotenv   # lee el archivo .env
from openai import OpenAI

load_dotenv()
cliente_openai = OpenAI()

embeddings_openai = []   # un vector de 1536 números por noticia
tokens_totales = 0

inicio = time.time()
for desde in range(0, len(textos), 500):
    lote_de_textos = textos[desde:desde + 500]

    respuesta = cliente_openai.embeddings.create(
        model="text-embedding-3-small",
        input=lote_de_textos,
    )

    # respuesta.data tiene un elemento por texto, en el mismo orden
    for elemento in respuesta.data:
        embeddings_openai.append(elemento.embedding)

    tokens_totales = tokens_totales + respuesta.usage.total_tokens

segundos_openai = time.time() - inicio
coste_usd = tokens_totales / 1_000_000 * 0.02

print(len(embeddings_openai), "vectores de", len(embeddings_openai[0]), "dimensiones")
print(f"Tokens usados: {tokens_totales:,}")
print(f"Coste: {coste_usd:.4f} USD")
print(f"Tiempo: {segundos_openai:.1f} s  (MiniLM en CPU: {segundos_embeddings:.1f} s)")

5000 vectores de 1536 dimensiones
Tokens usados: 269,721
Coste: 0.0054 USD
Tiempo: 12.1 s  (MiniLM en CPU: 6.9 s)


In [22]:
try:
    cliente.delete_collection("noticias_openai")
except Exception:
    pass

coleccion_openai = cliente.create_collection(
    name="noticias_openai",
    metadata={"hnsw:space": "cosine"},
)

for desde in range(0, len(textos), 500):
    hasta = desde + 500
    coleccion_openai.add(
        ids=ids[desde:hasta],
        documents=textos[desde:hasta],
        embeddings=embeddings_openai[desde:hasta],
        metadatas=metadatos[desde:hasta],
    )

print("Documentos en noticias_openai:", coleccion_openai.count())

Documentos en noticias_openai: 5000


Comparamos las **3 mejores** noticias de cada modelo con las mismas queries. La última está en español para ver cómo se defiende cada modelo con otro idioma.

In [23]:
queries_comparar = [
    "oil prices hit a record high",
    "a new computer virus is spreading on the internet",
    "el petróleo sube de precio",
]

for query in queries_comparar:
    print("🔎", query)

    # --- MiniLM
    query_embedding_minilm = model.encode([query])
    resultados_minilm = coleccion.query(query_embeddings=query_embedding_minilm, n_results=3)

    print("   MiniLM (384 dim):")
    for i in range(3):
        similitud = 1 - resultados_minilm["distances"][0][i]
        categoria = resultados_minilm["metadatas"][0][i]["categoria"]
        texto_corto = resultados_minilm["documents"][0][i][:70]
        print(f"      {similitud:.3f} [{categoria:8}] {texto_corto}")

    # --- OpenAI (la query también se convierte con OpenAI)
    respuesta_query = cliente_openai.embeddings.create(model="text-embedding-3-small", input=[query])
    query_embedding_openai = respuesta_query.data[0].embedding
    resultados_openai = coleccion_openai.query(query_embeddings=[query_embedding_openai], n_results=3)

    print("   OpenAI (1536 dim):")
    for i in range(3):
        similitud = 1 - resultados_openai["distances"][0][i]
        categoria = resultados_openai["metadatas"][0][i]["categoria"]
        texto_corto = resultados_openai["documents"][0][i][:70]
        print(f"      {similitud:.3f} [{categoria:8}] {texto_corto}")
    print()

🔎 oil prices hit a record high
   MiniLM (384 dim):
      0.848 [Business] Oil Prices Hit Record (Reuters) Reuters - Oil prices jumped to a new r
      0.843 [Business] Oil Above  #36;47 After New Record (Reuters) Reuters - Oil prices stru
      0.836 [Business] Oil Hits New Record as Demand Soars  LONDON (Reuters) - Oil prices str
   OpenAI (1536 dim):
      0.695 [Business] Oil Prices Rebound Towards Record Highs Oil prices moved back towards 
      0.683 [Business] Oil Prices Hit Record (Reuters) Reuters - Oil prices jumped to a new r
      0.667 [Business] Oil Hits New Record as Demand Soars  LONDON (Reuters) - Oil prices str

🔎 a new computer virus is spreading on the internet
   MiniLM (384 dim):
      0.610 [Sci/Tech]  #39;One in 12 Emails Infected with Virus #39; The number of attempted
      0.609 [Sci/Tech] New worm uses IM to lure victims A new version of the worm that spread
      0.597 [Sci/Tech] E-Mail Viruses Getting Smarter, Report Says SAN FRANCISCO (Reuters) - 
   Ope

## Bonus 2 · benchmark de inserción

Medimos cuánto tarda insertar las 5.000 noticias (con sus embeddings ya calculados) según el tamaño del batch: **1** (uno a uno), **100** y **1.000**.

Cada prueba usa una colección temporal que se borra al terminar, para empezar siempre desde cero. La función hace el mismo bucle del paso 5, pero con el tamaño de batch como parámetro.

In [24]:
def medir_insercion(tamano_batch):
    nombre_temporal = f"benchmark_{tamano_batch}"

    try:
        cliente.delete_collection(nombre_temporal)
    except Exception:
        pass
    coleccion_temporal = cliente.create_collection(name=nombre_temporal, metadata={"hnsw:space": "cosine"})

    inicio = time.time()
    for desde in range(0, len(textos), tamano_batch):
        hasta = desde + tamano_batch
        coleccion_temporal.add(
            ids=ids[desde:hasta],
            documents=textos[desde:hasta],
            embeddings=embeddings[desde:hasta],
            metadatas=metadatos[desde:hasta],
        )
    segundos = time.time() - inicio

    cliente.delete_collection(nombre_temporal)   # limpieza
    return segundos

In [25]:
tiempos = {}
for tamano_batch in [1, 100, 1000]:
    print(f"Probando batch de {tamano_batch}...")
    tiempos[tamano_batch] = medir_insercion(tamano_batch)

print()
print("Batch   Segundos   Docs/segundo")
for tamano_batch, segundos in tiempos.items():
    documentos_por_segundo = len(textos) / segundos
    print(f"{tamano_batch:>5}   {segundos:>8.2f}   {documentos_por_segundo:>10.0f}")

veces_mas_lento = tiempos[1] / tiempos[1000]
print(f"\nUno a uno es {veces_mas_lento:.0f} veces más lento que batches de 1000.")

Probando batch de 1...
Probando batch de 100...
Probando batch de 1000...

Batch   Segundos   Docs/segundo
    1      53.20           94
  100       4.09         1224
 1000       3.00         1665

Uno a uno es 18 veces más lento que batches de 1000.


## Bonus 3 · uso de espacio

¿Cuánto ocupa `./chroma_db`? Nuestra carpeta ahora contiene dos colecciones (`noticias` y `noticias_openai`), así que no sirve para medir una sola. Creamos una carpeta **aparte** solo con las 5.000 noticias de MiniLM, la medimos y la borramos.

`os.walk` recorre una carpeta y todas sus subcarpetas; sumamos el tamaño de cada archivo (`os.path.getsize`, en bytes).

In [26]:
def tamano_carpeta_mb(ruta):
    bytes_totales = 0
    for carpeta, subcarpetas, archivos in os.walk(ruta):
        for archivo in archivos:
            ruta_archivo = os.path.join(carpeta, archivo)
            bytes_totales = bytes_totales + os.path.getsize(ruta_archivo)
    return bytes_totales / (1024 * 1024)


RUTA_PRUEBA = "./chroma_db_tamano"
shutil.rmtree(RUTA_PRUEBA, ignore_errors=True)   # por si existe de una ejecución anterior

cliente_prueba = chromadb.PersistentClient(path=RUTA_PRUEBA)
coleccion_prueba = cliente_prueba.create_collection(name="noticias", metadata={"hnsw:space": "cosine"})

for desde in range(0, len(textos), 500):
    hasta = desde + 500
    coleccion_prueba.add(
        ids=ids[desde:hasta],
        documents=textos[desde:hasta],
        embeddings=embeddings[desde:hasta],
        metadatas=metadatos[desde:hasta],
    )

mb_real = tamano_carpeta_mb(RUTA_PRUEBA)
print(f"5.000 documentos ocupan {mb_real:.1f} MB en disco")

# Solo los vectores: 5000 filas x 384 números x 4 bytes (float32)
mb_solo_vectores = 5000 * 384 * 4 / (1024 * 1024)
print(f"   solo los vectores serían {mb_solo_vectores:.1f} MB")
print(f"   el resto (textos, metadatos, índice) son {mb_real - mb_solo_vectores:.1f} MB")

# Extrapolación lineal a 1 millón de documentos
gb_un_millon = mb_real / 5000 * 1_000_000 / 1024
print(f"\nExtrapolando a 1.000.000 de documentos: unos {gb_un_millon:.1f} GB")

# Si usáramos OpenAI (1536 dimensiones), los vectores pesarían 4 veces más
gb_vectores_openai = 1_000_000 * 1536 * 4 / (1024 ** 3)
print(f"Solo los vectores de OpenAI (1536 dim) para 1.000.000: {gb_vectores_openai:.1f} GB")

5.000 documentos ocupan 21.1 MB en disco
   solo los vectores serían 7.3 MB
   el resto (textos, metadatos, índice) son 13.8 MB

Extrapolando a 1.000.000 de documentos: unos 4.1 GB
Solo los vectores de OpenAI (1536 dim) para 1.000.000: 5.7 GB


In [27]:
# Limpieza: borramos la carpeta de prueba (la base real, ./chroma_db, se queda)
del cliente_prueba, coleccion_prueba
shutil.rmtree(RUTA_PRUEBA, ignore_errors=True)
print("Carpeta de prueba borrada. Tamaño de ./chroma_db (2 colecciones):", f"{tamano_carpeta_mb('./chroma_db'):.1f} MB")

Carpeta de prueba borrada. Tamaño de ./chroma_db (2 colecciones): 381.8 MB


## Reflexión

Conclusiones a partir de los números de este notebook (los tuyos pueden variar un poco según tu máquina).

- **Los filtros funcionan, pero no «entienden».** Con `categoria == Sports` ningún resultado era de otra categoría, y con `fecha >= 20250601` ninguno era anterior. Pero la query de bolsa forzada a `Sports` devolvió resultados con similitud 0.24-0.31: el filtro restringe, no valida que haya algo bueno.
- **Chroma y NumPy dan los mismos resultados.** Con 5.000 vectores, las 4 queries devolvieron los mismos 5 ids en el mismo orden. El índice aproximado (HNSW) no se nota con un corpus tan pequeño.
- **A esta escala, NumPy es más rápido por query.** 0.29 ms (NumPy) frente a 1.34 ms (Chroma). Con 5.000 vectores comparar con todos es casi gratis. Chroma compensa con lo que NumPy no tiene: persistencia, filtros por metadatos y un índice que escala a cientos de miles o millones sin comparar con todos.
- **Arrancar de nuevo.** Abrir la colección en un Python nuevo tardó 1.2 s (incluye importar `chromadb`), frente a 6.7 s de recalcular los embeddings. Aquí la diferencia es pequeña porque el corpus es pequeño y el modelo, ligero. Con un corpus grande o con embeddings de API (de pago), no tener que recalcular es el gran ahorro.
- **Batches.** Insertar uno a uno tardó 39.7 s; en batches de 100, 2.9 s; en batches de 1000, 2.5 s. Uno a uno es **16 veces más lento**. Pasar de 100 a 1000 casi no mejora: a partir de cierto tamaño ya no se gana nada.
- **MiniLM vs OpenAI.** Para estas queries en inglés los dos recuperan las mismas noticias, a veces en distinto orden. Embeber el corpus con OpenAI costó 0.0054 USD (269.721 tokens) y tardó 14 s, más que MiniLM en local (6.7 s). Las similitudes **no son comparables entre modelos**: con la query en español, MiniLM da 0.81 y OpenAI 0.50, y las dos recuperan noticias correctas. Lo que importa es el orden, no el número.
- **Espacio.** 5.000 noticias ocupan unos 21 MB (los vectores son solo 7.3 MB; el resto son textos, metadatos e índice). Extrapolando, 1 millón de documentos serían ~4 GB con MiniLM. Con OpenAI (1536 dimensiones) solo los vectores pesarían ~5.7 GB, y lo anterior se suma por encima.
- **Idea clave:** una base vectorial no hace la búsqueda «mejor» que NumPy; hace posible lo que NumPy no puede: guardar en disco, filtrar por metadatos, modificar documentos y escalar. Esta base (`./chroma_db`) es la que reutilizaremos en el primer pipeline RAG completo (SM3).